<a href="https://colab.research.google.com/github/shubham-dg10/AutomateChatGPTpromptswithPython/blob/main/stable/f222_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, requests, subprocess, time, re, atexit
from random import randint
from threading import Timer
from queue import Queue

# =========================================================
# 1. CLEAN ENVIRONMENT
# =========================================================

print("STEP 1: CLEANING ENVIRONMENT")

%cd /content
!rm -rf /content/*

# =========================================================
# 2. SYSTEM PACKAGES
# =========================================================

print("STEP 2: INSTALLING SYSTEM PACKAGES")

!apt -y update -qq
!apt -y install -qq aria2 git git-lfs unzip

# =========================================================
# 3. INSTALL COMFYUI
# =========================================================

print("STEP 3: INSTALLING COMFYUI")

!git clone https://github.com/comfyanonymous/ComfyUI

%cd /content/ComfyUI

!pip install -q -r requirements.txt

# =========================================================
# 4. AI DEPENDENCIES
# =========================================================

print("STEP 4: INSTALLING AI DEPENDENCIES")

!pip install -q \
torch \
torchvision \
torchaudio \
xformers \
mediapipe \
insightface \
piexif \
blake3 \
dynamicprompts \
ultralytics \
segment-anything \
onnxruntime-gpu \
onnxruntime \
opencv-python-headless \
accelerate \
safetensors \
einops

# =========================================================
# 5. CUSTOM NODES
# =========================================================

print("STEP 5: INSTALLING CUSTOM NODES")

%cd /content/ComfyUI/custom_nodes

repos = [
    "https://github.com/ltdrdata/ComfyUI-Manager",
    "https://github.com/cubiq/ComfyUI_IPAdapter_plus",
    "https://github.com/ltdrdata/ComfyUI-Impact-Pack",
    "https://github.com/cubiq/ComfyUI_Essentials",
    "https://github.com/Fannovel16/comfyui_controlnet_aux"
]

for repo in repos:
    folder = repo.split("/")[-1]

    if not os.path.exists(folder):
        !git clone {repo}

# =========================================================
# 6. MODEL FOLDERS
# =========================================================

print("STEP 6: CREATING MODEL DIRECTORIES")

!mkdir -p /content/ComfyUI/models/checkpoints
!mkdir -p /content/ComfyUI/models/ipadapter
!mkdir -p /content/ComfyUI/models/clip_vision

# =========================================================
# 7. DOWNLOAD JUGGERNAUT XL V9
# =========================================================

print("STEP 7: DOWNLOADING JUGGERNAUT XL V9")

!aria2c \
--console-log-level=error \
-c -x 16 -s 16 -k 1M \
"https://civitai.com/api/download/models/348913" \
-d /content/ComfyUI/models/checkpoints \
-o juggernautXL_v9.safetensors

# =========================================================
# 8. DOWNLOAD IPADAPTER FACEID SDXL
# =========================================================

print("STEP 8: DOWNLOADING IPADAPTER FACEID SDXL")

!aria2c \
--console-log-level=error \
-c -x 16 -s 16 -k 1M \
"https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin" \
-d /content/ComfyUI/models/ipadapter \
-o ip-adapter-faceid-plusv2_sdxl.bin

# =========================================================
# 9. DOWNLOAD CLIP VISION
# =========================================================

print("STEP 9: DOWNLOADING CLIP VISION")

%cd /content/ComfyUI/models/clip_vision

!wget -O CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors \
https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors

# =========================================================
# 10. DOWNLOAD INSIGHTFACE MODELS
# =========================================================

print("STEP 10: DOWNLOADING INSIGHTFACE MODELS")

!mkdir -p /root/.insightface/models

%cd /root/.insightface/models

!wget -O buffalo_l.zip \
https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip

!unzip -o buffalo_l.zip -d buffalo_l

# =========================================================
# 11. VERIFY FILES
# =========================================================

print("STEP 11: VERIFYING FILES")

print("\nCHECKPOINTS:")
!ls /content/ComfyUI/models/checkpoints

print("\nIPADAPTER:")
!ls /content/ComfyUI/models/ipadapter

print("\nCLIP VISION:")
!ls /content/ComfyUI/models/clip_vision

print("\nINSIGHTFACE:")
!ls /root/.insightface/models/buffalo_l

# =========================================================
# 12. CLOUDFLARED
# =========================================================

print("STEP 12: INSTALLING CLOUDFLARED")

if not os.path.exists('/content/cloudflared'):
    !wget -q \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /content/cloudflared

    !chmod 777 /content/cloudflared

# =========================================================
# 13. PUBLIC URL
# =========================================================

print("STEP 13: STARTING PUBLIC TUNNEL")

def tunnel(port, metrics_port, q):
    p = subprocess.Popen(
        [
            '/content/cloudflared',
            'tunnel',
            '--url',
            f'http://127.0.0.1:{port}',
            '--metrics',
            f'127.0.0.1:{metrics_port}'
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT
    )

    atexit.register(p.terminate)

    for _ in range(30):
        time.sleep(2)

        try:
            data = requests.get(
                f'http://127.0.0.1:{metrics_port}/metrics'
            ).text

            url = re.search(
                '(https?://.+?trycloudflare.com)',
                data
            ).group(1)

            if url:
                q.put(url)
                return

        except:
            pass

q = Queue()

Timer(
    2,
    tunnel,
    args=(8188, randint(8100, 9000), q)
).start()

public_url = q.get()

# =========================================================
# 14. START COMFYUI
# =========================================================

print("\n=================================================")
print("COMFYUI READY")
print("OPEN THIS URL:")
print(public_url)
print("WAIT 30-60 SECONDS")
print("=================================================\n")

%cd /content/ComfyUI

!python main.py \
--listen 0.0.0.0 \
--port 8188 \
--use-pytorch-cross-attention